# Parameter Sensitivity Explorer

This notebook performs a **vectorized grid search** using the `ggTrader` orchestrator api. It visualizes the profitability landscape to find robust parameter regions.

In [40]:
import sys
import os
import pandas as pd
import numpy as np
import vectorbt as vbt
import plotly.graph_objects as go
from tabulate import tabulate

# Auto-reload custom modules
%load_ext autoreload
%autoreload 2

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..',  'src'))
if project_root not in sys.path:
    sys.path.append(project_root)

from ggTrader.core.orchestrator import run_sensitivity_orchestrator

print("Environment initialized.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Environment initialized.


In [41]:
# --- Configuration ---
CONSTANTS = {
    "SYMBOLS": None,
    "SYMBOLS_FILE": os.path.join(os.getcwd(), "..", "data", "top_20_USD_1095_movers.json"),
    "START_DATE": "2023-01-01",
    "END_DATE": "2025-12-31",
    "INTERVAL": "4h",
    "START_CASH": 1000,
    "PORTFOLIO_SHARE": 0.10,
    "FEES": 0.005, # kraken max 0.4%
    "MIN_TRADES": 10,
    "SLIPPAGE": 0.003,
}

print("Configuration loaded.")

Configuration loaded.


In [42]:
# --- Define Parameter Grid ---
params = {
    # entry
    "sar_acceleration": [0.02],
    "sar_maximum": [0.2],
    "use_dmp_cross": [False],
    "adx_threshold": list(range(5, 35, 5)),
    "adx_length": list(range(5, 30, 5)),
    # exit
    "atr_length": list(range(5, 30, 5)), # varies a lot
    "atr_multiplier": list(np.arange(0.1, 1.0, 0.1)),  # small values work for some reason

}

print("Parameter grid defined.")

Parameter grid defined.


In [43]:
# --- Run Vectorized Analysis ---
# Note: show_progress=True enables VectorBT's tqdm progress bar
results = run_sensitivity_orchestrator(
    config=CONSTANTS, param_grid=params, save_results=False, show_progress=True
)

results_df = results["results_df"]
best_params = results["best_params"]

print("\nAnalysis Complete.")

Loading data...
Running Vectorized Sensitivity Analysis in 2 chunks (900 total combinations, chunk_size=500)...
  > Processing chunk 1 of 2 (0 to 500)...


  0%|          | 0/500 [00:00<?, ?it/s]

  > Processing chunk 2 of 2 (500 to 900)...


  0%|          | 0/400 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]


Analysis Complete.


In [44]:
top_10 = results_df.sort_values("Sharpe Ratio", ascending=False).head(10)
stats = top_10.agg(['mean', 'std', 'min', 'max']).round(3)
print("\nTOP 10 COMBINATIONS:")
print(tabulate(top_10, headers="keys", tablefmt="simple", showindex=False))

print("\n TOP 10 PARAMETER STATS")
print(tabulate(stats.T, headers='keys', tablefmt='simple', numalign="right"))



TOP 10 COMBINATIONS:
  adx_length    adx_threshold    sar_acceleration    sar_maximum  use_dmp_cross      atr_length    atr_multiplier    Sharpe Ratio
------------  ---------------  ------------------  -------------  ---------------  ------------  ----------------  --------------
          25               30                0.02            0.2  False                      15               0.5         1.70004
          25               30                0.02            0.2  False                      25               0.5         1.64935
          25               30                0.02            0.2  False                      20               0.5         1.62865
          25               30                0.02            0.2  False                      10               0.5         1.49578
          25               30                0.02            0.2  False                       5               0.5         1.42099
          25               25                0.02            0.2  Fa

In [45]:
# --- Visualization Helper ---
def show_heatmap(df, x_param, y_param, metric="Sharpe Ratio"):
    """Generates and displays a heatmap for the given parameter pair."""
    heatmap_data = df.pivot_table(
        index=y_param, 
        columns=x_param, 
        values=metric,
        aggfunc="mean"
    )

    fig = go.Figure(data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale='Viridis',
        colorbar=dict(title=metric)
    ))

    fig.update_layout(
        title=f"{metric} Landscape: {y_param} vs {x_param}",
        xaxis_title=x_param,
        yaxis_title=y_param
    )

    fig.show()

print("Visualization helper defined.")

Visualization helper defined.


In [46]:
# --- Generate Heatmaps ---
# You can list any pairs of parameters you want to explore
pairs_to_plot = [
    ("adx_threshold", "adx_length"), #entry
    # ("sar_acceleration", "sar_maximum"),
    ("atr_multiplier", "atr_length"), #exit
    ("adx_length", "atr_length"),
    # ("use_dmp_cross", "adx_length"),
    # ("use_dmp_cross", "adx_threshold")
]

for x, y in pairs_to_plot:
    show_heatmap(results_df, x, y)